# 4058. Maximum Pulse Value After One Subarray Rotation — Nhập môn Prefix Sum + "giữ giá trị tốt nhất phía trước"

Bài toán LeetCode (Medium):

```python
class Solution:
    def maxValue(self, nums: List[int]) -> int:
        ...
```

## Mục tiêu của notebook

Sau notebook này, bạn sẽ hiểu:

- **Tổng đan dấu** (alternating sum) là gì và cách tính bằng Python.
- Cách **xoay trái** một đoạn con bằng slicing.
- Lời giải **Brute Force** `O(n^3)` — dễ nghĩ nhất nhưng quá chậm.
- Cách phân tích **"thao tác làm thay đổi bao nhiêu?"** (tính độ chênh `Δ`) → lời giải `O(n^2)`.
- Kỹ thuật **Prefix Sum** (mảng tổng tiền tố) để tính tổng đoạn trong `O(1)`.
- Kỹ thuật **quét từ trái sang phải + giữ giá trị lớn nhất đã gặp** → lời giải `O(n)`.
- Trace từng bước, code chuẩn LeetCode, lỗi hay mắc, biến thể và bài tập có đáp án.

# 1. Đề bài nói gì?

Cho mảng số nguyên `nums` độ dài `n`.

**Pulse value** (giá trị xung) của một mảng `arr` là tổng **đan dấu** bắt đầu bằng dấu cộng:

```text
pulse(arr) = arr[0] - arr[1] + arr[2] - arr[3] + ...
```

Bạn được làm **nhiều nhất một** thao tác:

- Chọn `l < r`.
- **Xoay trái** đoạn `nums[l..r]` đúng 1 vị trí: phần tử đầu đoạn chạy xuống cuối đoạn.
  Ví dụ `[a, b, c, d]` → `[b, c, d, a]`.

Trả về pulse **lớn nhất** có thể đạt được.

### Ví dụ

| nums | Thao tác tốt nhất | Mảng sau | Pulse |
|---|---|---|---:|
| `[1, 5, 2]` | xoay `[0..1]`: `[1,5]→[5,1]` | `[5, 1, 2]` | `6` |
| `[6, 4, 3]` | xoay `[1..2]`: `[4,3]→[3,4]` | `[6, 3, 4]` | `7` |
| `[9, 7]` | không làm gì | `[9, 7]` | `2` |

### Ràng buộc

- `1 <= n <= 10^5`
- `-10^9 <= nums[i] <= 10^9` (có số âm!)

# 2. Nền tảng Python 1 — Tính tổng đan dấu

Phần tử ở chỉ số **chẵn** mang dấu `+`, chỉ số **lẻ** mang dấu `-`.

Để biết chẵn hay lẻ ta dùng phép chia lấy dư `%`:

- `i % 2 == 0` → `i` chẵn
- `i % 2 == 1` → `i` lẻ

Để vừa lấy chỉ số vừa lấy giá trị, dùng `enumerate`:

```python
for i, x in enumerate([10, 20, 30]):
    # i = 0, x = 10 ; i = 1, x = 20 ; i = 2, x = 30
```

In [1]:
for i, x in enumerate([10, 20, 30]):
    sign = 1 if i % 2 == 0 else -1
    print(f"i = {i}, x = {x}, dấu = {sign:+d}, đóng góp = {sign * x}")

i = 0, x = 10, dấu = +1, đóng góp = 10
i = 1, x = 20, dấu = -1, đóng góp = -20
i = 2, x = 30, dấu = +1, đóng góp = 30


In [2]:
def pulse(arr):
    total = 0
    for i, x in enumerate(arr):
        if i % 2 == 0:
            total += x
        else:
            total -= x
    return total


print(pulse([1, 5, 2]))    # 1 - 5 + 2 = -2
print(pulse([5, 1, 2]))    # 5 - 1 + 2 = 6
print(pulse([9, 7]))       # 2

-2
6
2


# 3. Nền tảng Python 2 — Xoay trái một đoạn bằng slicing

Nhắc lại slicing: `arr[a:b]` lấy các phần tử từ chỉ số `a` **đến trước** `b`.

Đoạn `nums[l..r]` (gồm cả `r`) viết là `nums[l:r+1]`.

Xoay trái 1 vị trí một list `sub`:

```python
sub[1:] + sub[:1]
# [a, b, c, d] -> [b, c, d] + [a] -> [b, c, d, a]
```

Ghép lại cả mảng:

```python
arr[:l] + (đoạn đã xoay) + arr[r+1:]
```

In [3]:
sub = ["a", "b", "c", "d"]
print(sub[1:], "+", sub[:1], "=", sub[1:] + sub[:1])

def rotate_left(arr, l, r):
    segment = arr[l:r + 1]
    rotated = segment[1:] + segment[:1]
    return arr[:l] + rotated + arr[r + 1:]

print(rotate_left([1, 5, 2], 0, 1))              # [5, 1, 2]
print(rotate_left([6, 4, 3], 1, 2))              # [6, 3, 4]
print(rotate_left([10, 20, 30, 40, 50], 1, 3))   # [10, 30, 40, 20, 50]

['b', 'c', 'd'] + ['a'] = ['b', 'c', 'd', 'a']
[5, 1, 2]
[6, 3, 4]
[10, 30, 40, 20, 50]


# 4. Cách 1 — Brute Force: thử mọi cặp `(l, r)`

Cách dễ nghĩ nhất:

1. Bắt đầu với `best = pulse(nums)` (trường hợp **không làm gì** — đề cho phép "nhiều nhất một").
2. Thử **mọi** cặp `l < r`: xoay, tính pulse, cập nhật `best`.

In [4]:
from typing import List

def max_pulse_bruteforce(nums: List[int]) -> int:
    n = len(nums)
    best = pulse(nums)                 # không làm thao tác nào
    for l in range(n):
        for r in range(l + 1, n):
            candidate = pulse(rotate_left(nums, l, r))
            best = max(best, candidate)
    return best


print(max_pulse_bruteforce([1, 5, 2]))   # 6
print(max_pulse_bruteforce([6, 4, 3]))   # 7
print(max_pulse_bruteforce([9, 7]))      # 2

6
7
2


## Độ phức tạp của Brute Force

- Số cặp `(l, r)`: khoảng `n^2 / 2`.
- Với mỗi cặp: tạo mảng mới và tính pulse mất `O(n)`.

→ Tổng `O(n^3)`. Với `n = 10^5`: khoảng `10^15 / 2` phép tính — không thể chạy kịp.

**Nút thắt:** với mỗi cặp, ta tính lại pulse **từ đầu**, dù phần lớn mảng không đổi. Câu hỏi đúng phải là:

> Xoay đoạn `[l..r]` làm pulse **thay đổi bao nhiêu**?

# 5. Quan sát: xoay trái làm dấu thay đổi thế nào?

Lấy `nums = [a0, a1, a2, a3, a4, a5]`, xoay đoạn `l = 1`, `r = 4`:

```text
Trước:  a0  a1  a2  a3  a4  a5
Sau:    a0  a2  a3  a4  a1  a5
```

Nhìn **dấu** của từng phần tử (chỉ số chẵn `+`, lẻ `-`):

| Phần tử | Chỉ số trước | Dấu trước | Chỉ số sau | Dấu sau | Thay đổi? |
|---|---:|:---:|---:|:---:|---|
| `a0` | 0 | + | 0 | + | không |
| `a1` | 1 | − | 4 | + | **đổi** (1 và 4 khác tính chẵn lẻ) |
| `a2` | 2 | + | 1 | − | **đổi** |
| `a3` | 3 | − | 2 | + | **đổi** |
| `a4` | 4 | + | 3 | − | **đổi** |
| `a5` | 5 | − | 5 | − | không |

### Hai quy luật

1. Các phần tử ở chỉ số `l+1 .. r` **lùi sang trái 1 ô** → chỉ số đổi tính chẵn lẻ → **dấu luôn đảo**.
2. Phần tử `nums[l]` **nhảy từ `l` đến `r`**:
   - nếu `l` và `r` **cùng** chẵn lẻ → dấu giữ nguyên;
   - nếu **khác** chẵn lẻ → dấu đảo.

Các phần tử ngoài đoạn không đổi gì.

In [5]:
def show_sign_changes(n, l, r):
    labels = [f"a{i}" for i in range(n)]
    after = rotate_left(labels, l, r)
    sign = lambda i: "+" if i % 2 == 0 else "-"
    new_index = {name: k for k, name in enumerate(after)}
    for i, name in enumerate(labels):
        j = new_index[name]
        changed = "ĐỔI" if sign(i) != sign(j) else ""
        print(f"{name}: chỉ số {i} ({sign(i)}) -> {j} ({sign(j)})  {changed}")

show_sign_changes(6, 1, 4)
print("-" * 40)
show_sign_changes(6, 1, 3)     # l, r cùng lẻ: a1 giữ nguyên dấu

a0: chỉ số 0 (+) -> 0 (+)  
a1: chỉ số 1 (-) -> 4 (+)  ĐỔI
a2: chỉ số 2 (+) -> 1 (-)  ĐỔI
a3: chỉ số 3 (-) -> 2 (+)  ĐỔI
a4: chỉ số 4 (+) -> 3 (-)  ĐỔI
a5: chỉ số 5 (-) -> 5 (-)  
----------------------------------------
a0: chỉ số 0 (+) -> 0 (+)  
a1: chỉ số 1 (-) -> 3 (-)  
a2: chỉ số 2 (+) -> 1 (-)  ĐỔI
a3: chỉ số 3 (-) -> 2 (+)  ĐỔI
a4: chỉ số 4 (+) -> 4 (+)  
a5: chỉ số 5 (-) -> 5 (-)  


Trong cell trên có hai cú pháp mới:

- `lambda i: ...` là **hàm vô danh** viết trên một dòng. `sign = lambda i: "+" if i % 2 == 0 else "-"` tương đương `def sign(i): return ...`.
- `{name: k for k, name in enumerate(after)}` là **dict comprehension**: tạo dict "tên → chỉ số mới".

# 6. Viết công thức độ chênh `Δ`

Gọi `s(t) = +1` nếu `t` chẵn, `-1` nếu `t` lẻ. Pulse là `Σ s(t) * nums[t]`.

Một phần tử đổi dấu từ `+x` sang `-x` (hoặc ngược lại) thì pulse thay đổi `-2 * (đóng góp cũ)`.

Vậy:

```text
Δ = -2 * Σ_{t = l+1..r} s(t) * nums[t]          (quy luật 1)
    + (s(r) - s(l)) * nums[l]                    (quy luật 2)
```

- Nếu `l`, `r` cùng chẵn lẻ: `s(r) - s(l) = 0`, phần thứ hai biến mất.
- Nếu khác chẵn lẻ: `s(r) - s(l) = -2 * s(l)`.

Pulse sau thao tác = `pulse(nums) + Δ`. Ta muốn **Δ lớn nhất** (và nếu mọi Δ đều âm thì chọn không làm gì: Δ = 0).

In [6]:
def delta_direct(nums, l, r):
    s = lambda t: 1 if t % 2 == 0 else -1
    part1 = -2 * sum(s(t) * nums[t] for t in range(l + 1, r + 1))
    part2 = (s(r) - s(l)) * nums[l]
    return part1 + part2

nums = [3, -2, 7, 1, -5, 4]
for l, r in [(0, 1), (1, 4), (2, 5), (0, 5)]:
    by_formula = pulse(nums) + delta_direct(nums, l, r)
    by_rotation = pulse(rotate_left(nums, l, r))
    print(f"(l={l}, r={r}): công thức = {by_formula:>3}, xoay thật = {by_rotation:>3}")

(l=0, r=1): công thức =  -8, xoay thật =  -8
(l=1, r=4): công thức =  -4, xoay thật =  -4
(l=2, r=5): công thức =   8, xoay thật =   8
(l=0, r=5): công thức =  -2, xoay thật =  -2


# 7. Cách 2 — Duyệt mọi `(l, r)` nhưng tính `Δ` tăng dần: `O(n^2)`

Cố định `l`, cho `r` chạy từ `l+1` sang phải. Tổng `Σ_{t = l+1..r} s(t)*nums[t]` **chỉ cần cộng thêm 1 số** mỗi khi `r` tăng 1.

Đây là ý "đừng tính lại từ đầu" — bước đầu tiên của tối ưu.

In [7]:
def max_pulse_quadratic(nums: List[int]) -> int:
    n = len(nums)
    s = [1 if t % 2 == 0 else -1 for t in range(n)]
    base = sum(s[t] * nums[t] for t in range(n))
    best_gain = 0                                   # không làm gì
    for l in range(n):
        running = 0                                 # Σ s(t)*nums[t] với t trong (l, r]
        for r in range(l + 1, n):
            running += s[r] * nums[r]
            gain = -2 * running + (s[r] - s[l]) * nums[l]
            best_gain = max(best_gain, gain)
    return base + best_gain


for nums in ([1, 5, 2], [6, 4, 3], [9, 7], [3, -2, 7, 1, -5, 4]):
    print(nums, "->", max_pulse_quadratic(nums), "| brute:", max_pulse_bruteforce(nums))

[1, 5, 2] -> 6 | brute: 6
[6, 4, 3] -> 7 | brute: 7
[9, 7] -> 2 | brute: 2
[3, -2, 7, 1, -5, 4] -> 20 | brute: 20


`O(n^2)` với `n = 10^5` vẫn là `5 * 10^9` phép tính — vẫn quá chậm. Ta cần bỏ hẳn vòng lặp trong.

Để làm vậy, trước hết cần một công cụ: **Prefix Sum**.

# 8. Nền tảng thuật toán — Prefix Sum (tổng tiền tố)

Ý tưởng: tính sẵn `P[t] = tổng của t phần tử đầu tiên`. Khi đó tổng một đoạn bất kỳ là **hiệu của hai giá trị P**:

```text
tổng(arr[a..b]) = P[b + 1] - P[a]
```

Ví dụ `arr = [4, 1, 3, 2]`:

| t | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| `P[t]` | 0 | 4 | 5 | 8 | 10 |

Tổng `arr[1..2] = 1 + 3 = 4 = P[3] - P[1] = 8 - 4`. ✔

> Quy ước `P[0] = 0` (tổng của 0 phần tử) giúp công thức đúng cả khi đoạn bắt đầu từ 0.

In [8]:
arr = [4, 1, 3, 2]
P = [0] * (len(arr) + 1)
for t in range(len(arr)):
    P[t + 1] = P[t] + arr[t]
print("P =", P)
print("tổng arr[1..2] =", P[3] - P[1])
print("tổng arr[0..3] =", P[4] - P[0])

P = [0, 4, 5, 8, 10]
tổng arr[1..2] = 4
tổng arr[0..3] = 10


Trong bài này ta dùng prefix sum của **dãy đã gắn dấu**:

```text
P[0] = 0
P[t + 1] = P[t] + s(t) * nums[t]
```

Nghĩa là `P[t]` = pulse của `t` phần tử đầu tiên. Và `P[n]` chính là pulse của cả mảng.

In [9]:
def signed_prefix(nums):
    P = [0] * (len(nums) + 1)
    for t, x in enumerate(nums):
        P[t + 1] = P[t] + (x if t % 2 == 0 else -x)
    return P

nums = [1, 5, 2]
print(signed_prefix(nums))     # [0, 1, -4, -2]  -> P[3] = -2 = pulse([1,5,2])

[0, 1, -4, -2]


# 9. Biến `Δ` thành hiệu của hai giá trị Prefix

Vì `Σ_{t = l+1..r} s(t) * nums[t] = P[r+1] - P[l+1]`:

**Trường hợp `l`, `r` cùng chẵn lẻ:**

```text
Δ = -2 * (P[r+1] - P[l+1]) = 2 * (P[l+1] - P[r+1])
```

**Trường hợp khác chẵn lẻ:** thêm `-2 * s(l) * nums[l]`:

```text
Δ = 2 * (P[l+1] - P[r+1]) - 2 * s(l) * nums[l]
  = 2 * (P[l+1] - s(l)*nums[l] - P[r+1])
  = 2 * (P[l] - P[r+1])                   vì P[l+1] - s(l)*nums[l] = P[l]
```

### Tóm tắt (rất quan trọng)

| Tính chẵn lẻ của `l` và `r` | `Δ(l, r)` |
|---|---|
| cùng | `2 * (P[l+1] - P[r+1])` |
| khác | `2 * (P[l] - P[r+1])` |

Điểm đẹp: `Δ` = (một thứ **chỉ phụ thuộc `l`**) − (một thứ **chỉ phụ thuộc `r`**). Đây là dấu hiệu để tối ưu xuống `O(n)`.

In [10]:
import random

def delta_prefix(P, l, r):
    if l % 2 == r % 2:
        return 2 * (P[l + 1] - P[r + 1])
    return 2 * (P[l] - P[r + 1])

random.seed(0)
for _ in range(300):
    nums = [random.randint(-9, 9) for _ in range(random.randint(2, 8))]
    P = signed_prefix(nums)
    for l in range(len(nums)):
        for r in range(l + 1, len(nums)):
            assert pulse(rotate_left(nums, l, r)) == P[-1] + delta_prefix(P, l, r)
print("Công thức Δ theo prefix khớp với xoay thật trên 300 mảng ngẫu nhiên ✔")

Công thức Δ theo prefix khớp với xoay thật trên 300 mảng ngẫu nhiên ✔


# 10. Cách 3 — Quét `r` từ trái sang phải, giữ "giá trị tốt nhất phía trước": `O(n)`

Cố định `r`. Ta cần `max` của `Δ(l, r)` với mọi `l < r`:

- Với `l` **cùng** chẵn lẻ với `r`: cần `max P[l+1]`.
- Với `l` **khác** chẵn lẻ với `r`: cần `max P[l]`.

Vì `P[r+1]` cố định khi đã chọn `r`, việc tối đa `Δ` chỉ còn là tìm **max của một đại lượng theo `l` trong quá khứ**.

Giống bài "mua bán cổ phiếu": khi đi qua từng ngày, ta luôn nhớ **giá thấp nhất đã gặp**. Ở đây ta nhớ **4 giá trị lớn nhất đã gặp**:

| Biến | Ý nghĩa |
|---|---|
| `best_same[0]` | max `P[l+1]` với các `l` **chẵn** đã gặp |
| `best_same[1]` | max `P[l+1]` với các `l` **lẻ** đã gặp |
| `best_diff[0]` | max `P[l]` với các `l` **chẵn** đã gặp |
| `best_diff[1]` | max `P[l]` với các `l` **lẻ** đã gặp |

Tại mỗi `r` (từ 1 đến n-1):

1. **Thêm** `l = r - 1` vào các biến "tốt nhất" (vì `l` phải `< r`).
2. `p = r % 2`. Ứng viên:
   - `2 * (best_same[p] - P[r+1])` (l cùng chẵn lẻ với r)
   - `2 * (best_diff[1 - p] - P[r+1])` (l khác chẵn lẻ với r)
3. Cập nhật `best_gain`.

Đáp án: `P[n] + best_gain`, với `best_gain` khởi tạo `0` (không làm gì).

`float('-inf')` là "âm vô cùng" — dùng làm giá trị khởi tạo cho max khi chưa có ứng viên nào. Mọi số đều lớn hơn nó.

In [11]:
NEG_INF = float("-inf")
print(NEG_INF < -10**18, max(NEG_INF, 5), 2 * (NEG_INF - 3))

True 5 -inf


In [12]:
def max_pulse_linear(nums: List[int]) -> int:
    n = len(nums)
    P = signed_prefix(nums)

    best_same = [NEG_INF, NEG_INF]   # max P[l+1], tách theo l chẵn / lẻ
    best_diff = [NEG_INF, NEG_INF]   # max P[l],   tách theo l chẵn / lẻ
    best_gain = 0                    # được phép không làm gì

    for r in range(1, n):
        l = r - 1                    # chỉ số l mới vừa trở thành hợp lệ
        best_same[l % 2] = max(best_same[l % 2], P[l + 1])
        best_diff[l % 2] = max(best_diff[l % 2], P[l])

        p = r % 2
        gain_same = 2 * (best_same[p] - P[r + 1])
        gain_diff = 2 * (best_diff[1 - p] - P[r + 1])
        best_gain = max(best_gain, gain_same, gain_diff)

    return P[n] + best_gain


for nums in ([1, 5, 2], [6, 4, 3], [9, 7], [5], [3, -2, 7, 1, -5, 4]):
    print(nums, "->", max_pulse_linear(nums))

[1, 5, 2] -> 6
[6, 4, 3] -> 7
[9, 7] -> 2
[5] -> 5
[3, -2, 7, 1, -5, 4] -> 20


Vì sao phép tính với `NEG_INF` vẫn an toàn? Khi chưa có `l` phù hợp, ứng viên là `2 * (-inf - P[r+1]) = -inf`, và `max(best_gain, -inf, ...)` sẽ bỏ qua nó. Không bao giờ chọn nhầm một `l` không tồn tại.

# 11. Mô phỏng từng bước

Ví dụ `nums = [1, 5, 2]`:

```text
P = [0, 1, -4, -2]      pulse ban đầu = P[3] = -2
```

| r | l mới thêm | best_same (chẵn, lẻ) | best_diff (chẵn, lẻ) | p = r%2 | gain_same | gain_diff | best_gain |
|---|---|---|---|---|---|---|---|
| 1 | 0 | (P[1]=1, −∞) | (P[0]=0, −∞) | 1 | 2(−∞ − P[2]) = −∞ | 2(0 − (−4)) = **8** | 8 |
| 2 | 1 | (1, P[2]=−4) | (0, P[1]=1) | 0 | 2(1 − (−2)) = 6 | 2(1 − (−2)) = 6 | 8 |

Đáp án `= -2 + 8 = 6` ✔ — ứng với `l = 0, r = 1` (khác chẵn lẻ): xoay `[1, 5]` thành `[5, 1]`.

In [13]:
def trace_max_pulse(nums):
    n = len(nums)
    P = signed_prefix(nums)
    print("nums =", nums)
    print("P    =", P, "| pulse ban đầu =", P[n])
    best_same = [NEG_INF, NEG_INF]
    best_diff = [NEG_INF, NEG_INF]
    best_gain, best_pair = 0, None
    for r in range(1, n):
        l = r - 1
        best_same[l % 2] = max(best_same[l % 2], P[l + 1])
        best_diff[l % 2] = max(best_diff[l % 2], P[l])
        p = r % 2
        gain_same = 2 * (best_same[p] - P[r + 1])
        gain_diff = 2 * (best_diff[1 - p] - P[r + 1])
        if max(gain_same, gain_diff) > best_gain:
            best_gain = max(gain_same, gain_diff)
            best_pair = r
        print(f"r={r}: best_same={best_same}, best_diff={best_diff}, "
              f"gain_same={gain_same}, gain_diff={gain_diff}, best_gain={best_gain}")
    print("Đáp án =", P[n], "+", best_gain, "=", P[n] + best_gain,
          "| r của thao tác tốt nhất:", best_pair)
    return P[n] + best_gain

trace_max_pulse([1, 5, 2])
print()
trace_max_pulse([6, 4, 3])

nums = [1, 5, 2]
P    = [0, 1, -4, -2] | pulse ban đầu = -2
r=1: best_same=[1, -inf], best_diff=[0, -inf], gain_same=-inf, gain_diff=8, best_gain=8
r=2: best_same=[1, -4], best_diff=[0, 1], gain_same=6, gain_diff=6, best_gain=8
Đáp án = -2 + 8 = 6 | r của thao tác tốt nhất: 1

nums = [6, 4, 3]
P    = [0, 6, 2, 5] | pulse ban đầu = 5
r=1: best_same=[6, -inf], best_diff=[0, -inf], gain_same=-inf, gain_diff=-4, best_gain=0
r=2: best_same=[6, 2], best_diff=[0, 6], gain_same=2, gain_diff=2, best_gain=2
Đáp án = 5 + 2 = 7 | r của thao tác tốt nhất: 2


7

In [14]:
# Một ví dụ "khó" hơn, có số âm và độ dài chẵn
trace_max_pulse([3, -2, 7, 1, -5, 4])
print("Brute force:", max_pulse_bruteforce([3, -2, 7, 1, -5, 4]))

nums = [3, -2, 7, 1, -5, 4]
P    = [0, 3, 5, 12, 11, 6, 2] | pulse ban đầu = 2
r=1: best_same=[3, -inf], best_diff=[0, -inf], gain_same=-inf, gain_diff=-10, best_gain=0
r=2: best_same=[3, 5], best_diff=[0, 3], gain_same=-18, gain_diff=-18, best_gain=0
r=3: best_same=[12, 5], best_diff=[5, 3], gain_same=-12, gain_diff=-12, best_gain=0
r=4: best_same=[12, 11], best_diff=[5, 12], gain_same=12, gain_diff=12, best_gain=12
r=5: best_same=[12, 11], best_diff=[11, 12], gain_same=18, gain_diff=18, best_gain=18
Đáp án = 2 + 18 = 20 | r của thao tác tốt nhất: 5
Brute force: 20


# 12. Lời giải LeetCode chuẩn

Bản nộp bài gọn hơn: không cần mảng `P` đầy đủ, chỉ cần `P[r]` và `P[r+1]` ở mỗi bước. Nhưng để dễ đọc, ta vẫn giữ mảng `P` (bộ nhớ `O(n)` hoàn toàn chấp nhận được).

Lưu ý: chữ ký LeetCode dùng `List[int]` nên cần `from typing import List`.

In [15]:
from typing import List

class Solution:
    def maxValue(self, nums: List[int]) -> int:
        n = len(nums)
        P = [0] * (n + 1)
        for t, x in enumerate(nums):
            P[t + 1] = P[t] + (x if t % 2 == 0 else -x)

        NEG = float("-inf")
        best_same = [NEG, NEG]    # max P[l+1] theo tính chẵn lẻ của l
        best_diff = [NEG, NEG]    # max P[l]   theo tính chẵn lẻ của l
        best_gain = 0

        for r in range(1, n):
            l = r - 1
            if P[l + 1] > best_same[l & 1]:
                best_same[l & 1] = P[l + 1]
            if P[l] > best_diff[l & 1]:
                best_diff[l & 1] = P[l]
            p = r & 1
            cand = 2 * (max(best_same[p], best_diff[p ^ 1]) - P[r + 1])
            if cand > best_gain:
                best_gain = cand

        return P[n] + best_gain

Mẹo nhỏ trong code nộp bài:

- `l & 1` bằng `l % 2` (lấy bit cuối) — nhanh hơn một chút.
- `p ^ 1` đổi `0 ↔ 1` — chính là `1 - p`.
- Gộp `max(best_same[p], best_diff[p ^ 1])` rồi trừ một lần, vì cả hai ứng viên cùng trừ `P[r+1]`.

In [16]:
sol = Solution()

tests = [
    ([1, 5, 2], 6),
    ([6, 4, 3], 7),
    ([9, 7], 2),
    ([5], 5),                          # n = 1: không có thao tác nào
    ([-3], -3),
    ([1, 2], 1),                       # xoay [1,2] -> [2,1]: 2 - 1 = 1
    ([-10**9, 10**9], 10**9 - (-10**9)),  # xoay -> [1e9, -1e9]
    ([0, 0, 0, 0], 0),
]

for nums, expected in tests:
    result = sol.maxValue(nums)
    print(f"{str(nums):28} -> {result:>12} | expected {expected:>12} | {'PASS' if result == expected else 'FAIL'}")
    assert result == expected

[1, 5, 2]                    ->            6 | expected            6 | PASS
[6, 4, 3]                    ->            7 | expected            7 | PASS
[9, 7]                       ->            2 | expected            2 | PASS
[5]                          ->            5 | expected            5 | PASS
[-3]                         ->           -3 | expected           -3 | PASS
[1, 2]                       ->            1 | expected            1 | PASS
[-1000000000, 1000000000]    ->   2000000000 | expected   2000000000 | PASS
[0, 0, 0, 0]                 ->            0 | expected            0 | PASS


In [17]:
# Đối chiếu ngẫu nhiên với Brute Force và O(n^2)
random.seed(2026)
for _ in range(2000):
    nums = [random.randint(-10, 10) for _ in range(random.randint(1, 9))]
    b = max_pulse_bruteforce(nums)
    assert max_pulse_quadratic(nums) == b
    assert sol.maxValue(nums) == b
print("3 cách cho cùng kết quả trên 2000 mảng ngẫu nhiên ✔")

3 cách cho cùng kết quả trên 2000 mảng ngẫu nhiên ✔


In [18]:
# Kiểm tra tốc độ với n = 10^5
import time
big = [random.randint(-10**9, 10**9) for _ in range(10**5)]
t0 = time.time()
sol.maxValue(big)
print(f"n = 100000 chạy trong {time.time() - t0:.3f} giây")

n = 100000 chạy trong 0.029 giây


# 13. Vì sao thuật toán đúng?

1. **Mọi khả năng đều được xét:** đáp án là `max(pulse(nums), max_{l<r} pulse sau khi xoay [l..r])`. Ta có `pulse sau = P[n] + Δ(l, r)`, và `best_gain = max(0, max Δ)`.
2. **Công thức `Δ` đúng:** dựa trên hai quy luật đổi dấu (mục 5) và đã kiểm chứng bằng code (mục 9).
3. **Quét đúng:** khi xử lý `r`, các biến `best_*` chứa đúng max trên **mọi** `l = 0..r-1` (bất biến của vòng lặp: mỗi vòng thêm đúng `l = r-1`). Vì `Δ(l, r) = f(l) - P[r+1]` với `f` là `P[l+1]` hoặc `P[l]` tuỳ chẵn lẻ, max theo `l` chính là max của `f(l)` trong nhóm chẵn lẻ phù hợp.

# 14. Độ phức tạp

Gọi `n = len(nums)`.

| Cách | Thời gian | Bộ nhớ phụ |
|---|---|---|
| 1. Brute Force | `O(n^3)` | `O(n)` |
| 2. Δ tăng dần | `O(n^2)` | `O(n)` (mảng dấu) |
| 3. Prefix + max phía trước | `O(n)` | `O(n)` cho `P` (có thể giảm còn `O(1)`) |

# 15. Những lỗi thường gặp

## Lỗi 1 — Quên trường hợp "không làm gì"

Đề nói **nhiều nhất** một thao tác. Nếu khởi tạo `best_gain = NEG_INF` thay vì `0` (tức là **bắt buộc** phải xoay), với `nums = [9, 7]` sẽ ra `7 - 9 = -2` thay vì `2`. Còn với `n = 1`, vòng lặp không chạy và kết quả là `-inf` — sai hoàn toàn.

## Lỗi 2 — Lệch chỉ số prefix

`P[t]` là tổng của `t` phần tử **đầu tiên** (chỉ số `0..t-1`). Tổng `nums[a..b]` là `P[b+1] - P[a]`, không phải `P[b] - P[a]`.

## Lỗi 3 — Quên phần tử `nums[l]` khi `l`, `r` khác chẵn lẻ

Chỉ đảo dấu đoạn `l+1..r` mà bỏ qua việc `nums[l]` cũng đổi dấu khi nhảy tới `r` → sai với ví dụ 1.

## Lỗi 4 — Cho phép `l = r`

Thật ra `l = r` không thay đổi gì (Δ = 0), nên không làm sai đáp án. Nhưng khi thêm `l` vào `best_*` **trước** khi tính cho `r`, phải thêm đúng `l = r - 1`, không phải `l = r`. Nếu thêm `l = r`, công thức "khác chẵn lẻ" có thể dùng nhầm.

In [19]:
def wrong_forced_rotation(nums):
    # Lỗi 1: bắt buộc phải xoay
    n = len(nums)
    best = float("-inf")
    for l in range(n):
        for r in range(l + 1, n):
            best = max(best, pulse(rotate_left(nums, l, r)))
    return best

print("Sai (buộc xoay):", wrong_forced_rotation([9, 7]))
print("Đúng           :", sol.maxValue([9, 7]))

Sai (buộc xoay): -2
Đúng           : 2


# 16. Mẹo nhận diện pattern

Dấu hiệu:

- "Làm **một** thao tác trên **một đoạn con** `[l..r]`, tối đa/tối thiểu một đại lượng".
- Đại lượng là **tổng** (có trọng số/dấu) → thay đổi chỉ xảy ra trong đoạn.

Quy trình:

1. Viết **Δ(l, r)** — thao tác làm đại lượng đổi bao nhiêu.
2. Dùng **prefix sum** để Δ chỉ còn vài giá trị `P[...]`.
3. Tách Δ thành `f(l) - g(r)` (hoặc `f(l) + g(r)`).
4. Quét `r`, **giữ max/min của `f(l)` với `l < r`** → `O(n)`.
5. Nếu Δ phụ thuộc tính chẵn lẻ / nhóm, giữ max **riêng cho từng nhóm**.

Các bài cùng họ: Best Time to Buy and Sell Stock, Maximum Subarray (Kadane), "đảo dấu một đoạn để tổng lớn nhất".

# 17. Biến thể — Hoán đổi hai phần tử **kề nhau** một lần

Nếu thao tác là "hoán đổi `nums[i]` và `nums[i+1]`" (nhiều nhất một lần):

```text
Δ(i) = s(i)*(nums[i+1] - nums[i]) + s(i+1)*(nums[i] - nums[i+1])
     = 2 * s(i) * (nums[i+1] - nums[i])       vì s(i+1) = -s(i)
```

Chỉ cần thử mọi `i` → `O(n)`. Thật ra đây chính là trường hợp `r = l + 1` của bài gốc (xoay đoạn dài 2 = hoán đổi).

In [20]:
def max_pulse_one_adjacent_swap(nums):
    base = pulse(nums)
    best_gain = 0
    for i in range(len(nums) - 1):
        s_i = 1 if i % 2 == 0 else -1
        best_gain = max(best_gain, 2 * s_i * (nums[i + 1] - nums[i]))
    return base + best_gain

print(max_pulse_one_adjacent_swap([1, 5, 2]))   # 6 (giống bài gốc)
print(max_pulse_one_adjacent_swap([3, -2, 7, 1, -5, 4]))

6
20


# 18. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `nums = [2, 8, 1, 4]`:

1. Tính `pulse(nums)`.
2. Viết mảng sau khi xoay trái `[0..3]`, `[1..3]`, `[0..2]` và tính pulse từng mảng.
3. Lập mảng `P` và kiểm tra công thức `Δ` của mục 9 cho ba cặp trên.

## Bài 2 — Dự đoán output

```python
rotate_left([1, 2, 3, 4, 5], 2, 4)
rotate_left([1, 2, 3, 4, 5], 0, 1)
signed_prefix([4, 4, 4])
```

In [21]:
print(rotate_left([1, 2, 3, 4, 5], 2, 4))
print(rotate_left([1, 2, 3, 4, 5], 0, 1))
print(signed_prefix([4, 4, 4]))

[1, 2, 4, 5, 3]
[2, 1, 3, 4, 5]
[0, 4, 0, 4]


## Bài 3 — Điền chỗ trống: tính tổng đoạn bằng prefix

In [22]:
def exercise_3_range_sum(arr, queries):
    """queries là list các cặp (a, b); trả về list tổng arr[a..b] (gồm cả b)."""
    # TODO 1: dựng P với P[0] = 0, P[t+1] = P[t] + arr[t]
    # TODO 2: với mỗi (a, b), tổng = P[b + 1] - P[a]
    raise NotImplementedError

# assert exercise_3_range_sum([4, 1, 3, 2], [(1, 2), (0, 3), (2, 2)]) == [4, 10, 3]

## Bài 4 — Tự viết lời giải `O(n)`

Không nhìn lại mục 10, hãy tự viết hàm dùng 4 biến `best_same`, `best_diff`.

In [23]:
def exercise_4_max_pulse(nums):
    # TODO
    raise NotImplementedError

## Bài 5 — Biến thể: tối thiểu pulse

Đổi đề: tìm pulse **nhỏ nhất** sau nhiều nhất một thao tác xoay trái. Cần đổi gì trong lời giải?

In [24]:
def exercise_5_min_pulse(nums):
    # TODO
    raise NotImplementedError

## Bài 6 — Mini challenge: xoay **phải** một vị trí

Thao tác mới: xoay **phải** đoạn `[l..r]`: `[a, b, c, d] → [d, a, b, c]`. Tìm pulse lớn nhất trong `O(n)`.

Gợi ý: làm lại mục 5–9 cho phép xoay mới.

In [25]:
def rotate_right(arr, l, r):
    segment = arr[l:r + 1]
    return arr[:l] + segment[-1:] + segment[:-1] + arr[r + 1:]

def exercise_6_max_pulse_right_rotation(nums):
    # TODO
    raise NotImplementedError

print(rotate_right([1, 2, 3, 4, 5], 1, 3))   # [1, 4, 2, 3, 5]

[1, 4, 2, 3, 5]


## Gợi ý theo mức độ

**Bài 4**
- Mức 1: Δ = `f(l) - P[r+1]`, với `f(l)` là `P[l+1]` (cùng chẵn lẻ) hoặc `P[l]` (khác).
- Mức 2: trước khi xét `r`, thêm `l = r - 1` vào max.
- Mức 3: `best_gain` khởi tạo `0`.

**Bài 5**
- Mức 1: tìm Δ **nhỏ nhất** thay vì lớn nhất.
- Mức 2: Δ nhỏ nhất = `2*(min f(l) - P[r+1])` → giữ **min** thay vì max.
- Mức 3: khởi tạo bằng `+inf`, `best_gain = 0` (không làm gì).

**Bài 6**
- Mức 1: các phần tử `l..r-1` dịch sang phải 1 ô → đổi dấu. `nums[r]` nhảy về `l`.
- Mức 2: Δ = `-2*(P[r] - P[l]) + (s(l) - s(r))*nums[r]`.
- Mức 3: cùng chẵn lẻ: `2*(P[l] - P[r])`; khác: `2*(P[l] - P[r+1])`. Chỉ cần max `P[l]` theo nhóm chẵn lẻ của `l`.

# 19. Đáp án tham khảo

**Bài 1:** `nums = [2, 8, 1, 4]`, `P = [0, 2, -6, -5, -9]`, pulse = `-9`.

| (l, r) | Mảng sau | Pulse | Cùng chẵn lẻ? | Δ theo công thức |
|---|---|---:|---|---|
| (0, 3) | `[8, 1, 4, 2]` | `8 - 1 + 4 - 2 = 9` | khác | `2*(P[0] - P[4]) = 2*(0 + 9) = 18` → `-9 + 18 = 9` ✔ |
| (1, 3) | `[2, 1, 4, 8]` | `2 - 1 + 4 - 8 = -3` | cùng | `2*(P[2] - P[4]) = 2*(-6 + 9) = 6` → `-3` ✔ |
| (0, 2) | `[8, 1, 2, 4]` | `8 - 1 + 2 - 4 = 5` | cùng | `2*(P[1] - P[3]) = 2*(2 + 5) = 14` → `5` ✔ |

**Bài 2:** `[1, 2, 4, 5, 3]`, `[2, 1, 3, 4, 5]`, `[0, 4, 0, 4]`.

In [26]:
def answer_3_range_sum(arr, queries):
    P = [0] * (len(arr) + 1)
    for t in range(len(arr)):
        P[t + 1] = P[t] + arr[t]
    return [P[b + 1] - P[a] for a, b in queries]

assert answer_3_range_sum([4, 1, 3, 2], [(1, 2), (0, 3), (2, 2)]) == [4, 10, 3]
print("Đáp án bài 3 PASS")

Đáp án bài 3 PASS


In [27]:
def answer_4_max_pulse(nums):
    n = len(nums)
    P = signed_prefix(nums)
    best_same = [NEG_INF, NEG_INF]
    best_diff = [NEG_INF, NEG_INF]
    best_gain = 0
    for r in range(1, n):
        l = r - 1
        best_same[l % 2] = max(best_same[l % 2], P[l + 1])
        best_diff[l % 2] = max(best_diff[l % 2], P[l])
        p = r % 2
        best_gain = max(best_gain,
                        2 * (best_same[p] - P[r + 1]),
                        2 * (best_diff[1 - p] - P[r + 1]))
    return P[n] + best_gain

random.seed(4)
for _ in range(500):
    nums = [random.randint(-9, 9) for _ in range(random.randint(1, 8))]
    assert answer_4_max_pulse(nums) == max_pulse_bruteforce(nums)
print("Đáp án bài 4 PASS")

Đáp án bài 4 PASS


In [28]:
POS_INF = float("inf")

def answer_5_min_pulse(nums):
    n = len(nums)
    P = signed_prefix(nums)
    low_same = [POS_INF, POS_INF]    # min P[l+1]
    low_diff = [POS_INF, POS_INF]    # min P[l]
    best_gain = 0                    # Δ nhỏ nhất (không làm gì -> 0)
    for r in range(1, n):
        l = r - 1
        low_same[l % 2] = min(low_same[l % 2], P[l + 1])
        low_diff[l % 2] = min(low_diff[l % 2], P[l])
        p = r % 2
        best_gain = min(best_gain,
                        2 * (low_same[p] - P[r + 1]),
                        2 * (low_diff[1 - p] - P[r + 1]))
    return P[n] + best_gain

def min_pulse_bruteforce(nums):
    best = pulse(nums)
    for l in range(len(nums)):
        for r in range(l + 1, len(nums)):
            best = min(best, pulse(rotate_left(nums, l, r)))
    return best

random.seed(5)
for _ in range(500):
    nums = [random.randint(-9, 9) for _ in range(random.randint(1, 8))]
    assert answer_5_min_pulse(nums) == min_pulse_bruteforce(nums)
print("Đáp án bài 5 PASS")

Đáp án bài 5 PASS


In [29]:
def answer_6_max_pulse_right_rotation(nums):
    n = len(nums)
    P = signed_prefix(nums)
    best_P = [NEG_INF, NEG_INF]      # max P[l] theo tính chẵn lẻ của l, với l < r
    best_gain = 0
    for r in range(1, n):
        l = r - 1
        best_P[l % 2] = max(best_P[l % 2], P[l])
        p = r % 2
        same = 2 * (best_P[p] - P[r])          # l cùng chẵn lẻ với r
        diff = 2 * (best_P[1 - p] - P[r + 1])  # l khác chẵn lẻ với r
        best_gain = max(best_gain, same, diff)
    return P[n] + best_gain

def max_pulse_right_bruteforce(nums):
    best = pulse(nums)
    for l in range(len(nums)):
        for r in range(l + 1, len(nums)):
            best = max(best, pulse(rotate_right(nums, l, r)))
    return best

random.seed(6)
for _ in range(1000):
    nums = [random.randint(-9, 9) for _ in range(random.randint(1, 8))]
    assert answer_6_max_pulse_right_rotation(nums) == max_pulse_right_bruteforce(nums)
print("Đáp án bài 6 PASS")

Đáp án bài 6 PASS


# 20. Tổng kết

## Kiến thức Python

```python
enumerate(arr)              # vừa chỉ số vừa giá trị
i % 2, i & 1, p ^ 1         # chẵn/lẻ, đổi 0 <-> 1
arr[l:r+1], sub[1:] + sub[:1]   # cắt và xoay
float("-inf"), float("inf") # giá trị khởi tạo cho max / min
lambda, dict comprehension
```

## Kỹ thuật thuật toán

1. **Tính Δ** thay vì tính lại toàn bộ.
2. **Prefix Sum**: tổng đoạn = hiệu hai giá trị prefix.
3. **Tách Δ = f(l) − g(r)** rồi **quét r, giữ max f(l) phía trước**.
4. Tách nhóm theo **tính chẵn lẻ** khi công thức phụ thuộc chẵn lẻ.

## Công thức nhớ nhanh

| `l`, `r` | Δ |
|---|---|
| cùng chẵn lẻ | `2*(P[l+1] - P[r+1])` |
| khác chẵn lẻ | `2*(P[l] - P[r+1])` |

Đáp án = `P[n] + max(0, max Δ)`.

## Độ phức tạp

```text
Time:  O(n)
Space: O(n)  (có thể O(1))
```

## Checklist trước khi nộp

- [ ] Có trường hợp **không làm gì** (`best_gain = 0`)?
- [ ] `n = 1` chạy đúng (vòng lặp không chạy)?
- [ ] Chỉ số prefix `P[l]`, `P[l+1]`, `P[r+1]` đúng?
- [ ] Có `from typing import List`?

# 21. Phiên bản cuối cùng nên nhớ

```python
from typing import List

class Solution:
    def maxValue(self, nums: List[int]) -> int:
        n = len(nums)
        P = [0] * (n + 1)
        for t, x in enumerate(nums):
            P[t + 1] = P[t] + (x if t % 2 == 0 else -x)

        NEG = float("-inf")
        best_same = [NEG, NEG]    # max P[l+1]
        best_diff = [NEG, NEG]    # max P[l]
        best_gain = 0

        for r in range(1, n):
            l = r - 1
            best_same[l % 2] = max(best_same[l % 2], P[l + 1])
            best_diff[l % 2] = max(best_diff[l % 2], P[l])
            p = r % 2
            best_gain = max(best_gain,
                            2 * (best_same[p] - P[r + 1]),
                            2 * (best_diff[1 - p] - P[r + 1]))

        return P[n] + best_gain
```

Khi hiểu được vì sao Δ tách được thành `f(l) − P[r+1]`, bạn đã nắm kỹ thuật dùng cho rất nhiều bài "một thao tác trên đoạn con".